# 你的第一个实验（Lab）

### 请先读完本节。即使偏长，也能帮你把后面几周的学习姿势摆正。

### 另外请务必阅读 [README.md](../README.md)！里面有更新视频与更多说明；课程资源入口也在这里：[紫色课程资源页](https://edwarddonner.com/2024/11/13/llm-engineering-resources/)

## 你的第一个「前沿大模型」小项目

课程结束时，你会做出由 **7 个智能体（Agents）** 协作解决业务问题的自主方案。现在我们从更小的事情起步。

目标：写一种「会摘要的浏览器」——给它一个 URL，它返回网页摘要。像「互联网读者文摘」。

开始前，请先完成 README 里链接的环境搭建。

### 若你刚接触 Notebook（也叫 Lab / Jupyter Lab）

欢迎来到数据科学实验环境！点选下方含代码的「单元格（cell）」，按 **Shift+Enter** 即可运行。请从上到下按顺序执行。

更多入门说明见 [Guides 文件夹](../guides/01_intro.ipynb)。

## 需要帮助时

有问题请联系课程方：平台留言、邮件 ed@edwarddonner.com，或 LinkedIn：https://www.linkedin.com/in/eddonner/  
作者也在尝试 X：[@edwarddonner](https://x.com/edwarddonner)

## 更多排错

请看安装目录里的 [故障排除](../setup/troubleshooting.ipynb) 笔记本；文末有诊断脚本与调试信息。

## 若这些对你已是「旧帽子」

若你已熟悉今天内容，仍建议过一遍：前几周可较快完成，后面会加深。最终我们会微调自己的 LLM，去和 OpenAI 等产品比一比！

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读 — 重要说明</h2>
            <span style="color:#900;">本课授课方式可能和其他课不同：讲师不会在你盯着屏幕时逐行敲代码，而是像这样跑 Jupyter Lab，让你对流程建立直觉。建议是：<b>先看完讲座</b>，再自己仔细跑一遍。多加 <code>print</code> 看清中间值，再改出你自己的变体。若有 GitHub，用仓库展示变体——既是练习，也能向未来客户或雇主证明能力……</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">代码是活资源 — 请留意公告邮件</h2>
            <span style="color:#f71;">作者会定期推送代码更新：有人提问时会加示例或改进注释。因此笔记本与视频可能不完全一致——视频里的内容都在这里，另外还加了更好的解释和新模型（例如 DeepSeek）。把它当成互动书。<br/><br/>重要更新常见于 Udemy 左侧「公告」；也可在通知设置里订阅邮件。作者会尽量只发有价值的邮件。
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">这些练习的商业价值</h2>
            <span style="color:#181;">笔记本既要好学，也尽量有趣（例如让模型讲笑话、互相抬杠）。但底层目标是可落地的商业技能。听课过程中会点出业务影响：你积累模型与技术经验时，请思考如何用到真实场景。想讨论想法，欢迎联系作者。</span>
        </td>
    </tr>
</table>


### 如需安装 Cursor 扩展

1. 打开「查看」菜单 →「扩展」
2. 搜索 Python
3. 安装 Microsoft 的 **Python**（`ms-python`），若尚未安装
4. 再搜索 Jupyter
5. 安装 Microsoft 的 **Jupyter**（`ms-toolsai`），若尚未安装

### 接着选择内核（Kernel）

点击右上角「选择内核」→「Python 环境...」  
选类似 `.venv (Python 3.12.x) .venv/bin/python` 的推荐项（常带星标）。

有问题？去故障排除笔记本。

### 注意：每个笔记本都要单独选一次内核。


In [56]:
# ========== 导入：本实验用到的库 ==========

# 导入标准库 os：读环境变量（例如 ANTHROPIC_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从本地 scraper 导入 fetch_website_contents：抓取/渲染网页正文（本贡献版配合 Playwright）
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown 与 display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display
# 从 litellm 导入 completion：统一接口调用多家 LLM 提供商（本实验走 Anthropic Claude）
from litellm import completion

# 若本格报错：请先打开故障排除笔记本排查环境与依赖


# 连接到 OpenAI（或 Ollama）——本笔记本实际用 Anthropic + LiteLLM

下一格会从 `.env` 加载环境变量，并检查 Anthropic API Key。

若想改用免费的 **Ollama**，请看 README 里「付费 API 的免费替代方案」；不确定时，解决方案文件夹里有完整示例（如 `day1_with_ollama.ipynb`）。

## 遇到问题怎么排

- 出现 `NameError`：是否从上到下跑过所有单元格？可参考 Python 基础指南里定位未定义名字的方法。
- 仍不行：打开 [故障排除](../setup/troubleshooting.ipynb)，按步骤找根因。
- 或联系作者：ed@edwarddonner.com

担心 API 费用？README 有说明——本课调用成本通常很低，你也可随时改用 Ollama（第 2 天会讲）。


In [ ]:
# ========== 加载环境变量并检查 Anthropic API Key ==========

# 加载名为 .env 的文件；override=True 表示用文件里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取 Anthropic 密钥（变量名必须是 ANTHROPIC_API_KEY，勿改）
api_key = os.getenv('ANTHROPIC_API_KEY')

# 以下分支只做「密钥形态」自检，方便新手定位配置问题（打印文案保持英文原样，便于对照排错文档）

# 情况 1：根本没读到密钥
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 情况 2：读到了，但前缀不像 Anthropic 密钥（常见 sk-ant-）
elif not api_key.startswith("sk-ant-"):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using the right key - see troubleshooting notebook")
# 情况 3：首尾可能有空格/制表符（从网页复制时常见）
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# 情况 4：形态看起来正常
else:
    print("API key found and looks good so far!")


# 先快速调用一次前沿模型（Frontier Model），当作预热预览！


In [ ]:
# ========== 构造第一条 messages：预览 LiteLLM / Anthropic 调用形状 ==========

# 用户要发给模型的纯文本（保留英文原文）
message = "Hello, Anthropic! This is my first ever message to you! Hi!"

# Chat API 的标准结构：列表里每项是 {role, content}；这里只有一条 user
messages = [{"role": "user", "content": message}]

# 在笔记本里直接写变量名：会显示该列表，便于确认结构
messages


In [ ]:
# ========== 真正发出第一次 completion 请求 ==========

# 用 litellm.completion 调用模型；model id 保持原样，供 LiteLLM 路由到对应提供商
response = completion(model="claude-sonnet-4-5-20250929", messages=messages)
# 取出第一条 choice 的助手回复正文（和 OpenAI SDK 的字段形状兼容）
response.choices[0].message.content


## 开始我们的第一个项目：URL → 网页摘要


In [ ]:
# ========== 试用 scraper：热重载后抓取个人站点 ==========

# importlib：开发时改完 scraper.py 可 reload，不必重启整个内核
import importlib
# 导入 scraper 包/模块（本格先 import 再 reload）
import scraper
# 重新加载磁盘上的最新 scraper 代码
importlib.reload(scraper)
# 再次从（可能已更新的）模块里导入抓取函数
from scraper import fetch_website_contents

# await 异步抓取指定 URL 的正文；把结果存进变量 ed（URL 保持原样）
ed = await fetch_website_contents("https://sneha-rao.de/")
# 打印抓到的文本，确认抓取是否成功、内容是否可读
print(ed)


## 提示类型（Prompt Types）

你可能已经知道——若还不知道，接下来会非常熟悉！

像 GPT / Claude 这类模型，训练时就按特定「消息角色」接收指令。它们通常期望：

- **系统提示（system prompt）**：任务是什么、语气如何
- **用户提示（user prompt）**：对话的起点 / 具体要处理的内容


In [29]:
# ========== 定义 system prompt：尖酸幽默的网页分析助手 ==========

# 多行字符串：角色 + 忽略导航噪声 + 用 markdown 回复且不要包代码块（英文指令保留，改译会改变风格）
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [30]:
# ========== 定义 user prompt 前缀：后面会拼接网页正文 ==========

# 告诉模型「下面是网页内容，请短摘要；若有新闻/公告也一并总结」（英文保留）
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## 消息（Messages）结构

OpenAI 等 API 期望特定结构的消息列表；很多其他 API（含 LiteLLM 统一接口）也共用这套形状：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下面 2 个单元格先做一个很简单的调用——还不会上「完整网页摘要」那套（那是预览）。


In [ ]:
# ========== 最小可运行示例：system + user 问一道算术 ==========

# 构造两条消息：system 定「乐于助人」，user 问 2+2（字符串保持英文原样）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 调用同一 Claude 模型；观察返回是否为简洁数字/句子
response = completion(model="claude-sonnet-4-5-20250929", messages=messages)
# 打印助手回复正文
response.choices[0].message.content


## 用函数为模型组装「有用」的 messages（网页摘要版）


In [32]:
# ========== messages_for：把网页正文装进标准 Chat messages ==========

# 这个函数产出的格式，与上面「system + user」示例完全一致，只是内容换成摘要任务
def messages_for(website):
    # system 用全局 system_prompt；user = 前缀说明 + 网页文本
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# 试一下：用刚才抓到的 ed 组装 messages，看看结构长什么样；也可换别的网站文本再试
messages_for(ed)


## 拼起来：抓取 → 组 messages → 调用 API


In [61]:
# ========== summarize：给定 URL，返回模型生成的 Markdown 摘要 ==========

# 异步函数：内部要 await 抓取；API 调用本身用同步 completion（与原逻辑一致）
async def summarize(url):
    # 先抓网页正文
    website = await fetch_website_contents(url)
    # 再调用 Claude；messages 由 messages_for 生成
    response = completion(
        model = "claude-sonnet-4-5-20250929",
        messages = messages_for(website)
    )
    # 只返回助手文本，方便上层展示
    return response.choices[0].message.content


In [35]:
# 对课程作者站点做一次摘要（返回字符串；本格未 display Markdown）
summarize("https://edwarddonner.com")


'# Edward Donner: Another LLM Evangelist Who Made It Your Problem\n\nOh look, it\'s Ed - a self-proclaimed LLM enthusiast who got so insufferable that his **friends literally begged him to stop talking and make Udemy courses instead**. Plot twist: 400,000 people actually signed up. Apparently turning your dinner party monologues into online lectures is a viable business model.\n\n## What\'s He About?\n- Co-founder/CTO of **Nebula.io** (applying AI to help people "discover their potential" - because that\'s not vague at all)\n- Previously founded **untapt** (acquired 2021 - at least someone wanted it)\n- Creates "top-rated" AI courses that are totally not just his friends trying to get him to shut up\n- Also makes "very amateur" electronic music (his words, refreshingly honest)\n\n## Recent Announcements\n- **Jan 2026**: AI Builder with n8n course (agents and voice agents, because of course)\n- **Nov 2025**: Something about "The Unique Energy of an AI Live Event" (?)\n- **Sept 2025**: M

In [62]:
# ========== display_summary：摘要后再用 Markdown 渲染到笔记本 ==========

# 异步封装：先 await summarize，再用 IPython display 展示
async def display_summary(url):
    # 拿到模型返回的 markdown 文本
    summary = await summarize(url)
    # 在笔记本输出区渲染为富文本，而不是裸字符串
    display(Markdown(summary))


In [ ]:
# 对自己的站点跑一遍完整链路：抓取 → 提示 → Claude → Markdown 展示
await display_summary("https://sneha-rao.de/")


# 再试更多网站

注意：这种简单抓取**只适合**能直接拿到可读 HTML/正文的站点。

用 JavaScript 渲染的站点（例如许多 React 应用）可能抓不到有效正文。社区贡献文件夹里有 Selenium 等方案；安装方式可自行查文档或问 ChatGPT。

另外，受 CloudFront 等防护的站点可能返回 403（感谢 Andy J 指出）。

不过很多站点仍然可以正常工作！


In [65]:
# 对新闻站 CNN 试摘要（页面大、JS 多，耗时与成功率因网络/渲染而异）
await display_summary("https://cnn.com")


Fetching website contents...


python(49092) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Navigating to URL: https://cnn.com
get_rendered_html Fetched HTML length: 5872195 <!DOCTYPE html><html lang="en" data-uri="cms.cnn.com/_pages/clg35wfph000047qb0ndy7s77@published" data-layout-uri="cms.cnn.com/_layouts/layout-homepage/instances/homepage-international@published" class="userconsent-cntry-de userconsent-state-be userconsent-reg-gdpr"><head>
<link rel="dns-prefetch" href="//tpc.googlesyndication.com">

<link rel="preconnect" href="//tpc.googlesyndication.com">

<link rel="dns-prefetch" href="//pagead2.googlesyndication.com">

<link rel="preconnect" href="//pagead2.
Closing browser
Fetched HTML length: 5872195 <!DOCTYPE html><html lang="en" data-uri="cms.cnn.com/_pages/clg35wfph000047qb0ndy7s77@published" data-layout-uri="cms.cnn.com/_layouts/layout-homepage/instances/homepage-international@published" class="userconsent-cntry-de userconsent-state-be userconsent-reg-gdpr"><head>
<link rel="dns-prefetch" href="//tpc.googlesyndication.com">

<link rel="preconnect" href="//tpc.go

# CNN Website Summary

This appears to be CNN's homepage with current breaking news stories. Here are the main headlines:

## Major News Stories:

**Minnesota Killing Backlash** - Growing controversy over a killing in Minnesota, with the White House showing "first signs of retreat." Tom Homan (White House border czar) is being deployed to Minnesota, and there's a controversial Border Patrol chief being sidelined.

**International News:**
- **India-EU Trade Deal** - India and EU have finalized a landmark trade pact described as the "mother of all deals"
- **Venezuela** - CNN Exclusive: US planning CIA presence in post-Maduro Venezuela
- **NATO** - NATO chief warns Europe can't defend itself without US support

**Other Stories:**
- US winter storm coverage
- Snow leopard attack at ski resort
- **Music:** Sly Dunbar, legendary reggae drummer, dies at 73
- **Entertainment:** K-pop idol reinvention story, Nigella Lawson mention, Paris Men's Fashion Week
- **Viral:** Year of the Horse toy goes viral in China

The site includes live updates, video content, and multiple exclusive CNN reports focusing heavily on US domestic issues and international relations.

In [39]:
# 再次调用个人站点摘要（注意：此处未写 await，与原笔记本一致——在部分环境下可能返回 coroutine 对象）
display_summary("https://sneha-rao.de/")


# Sneha Rao's Portfolio: Now Loading... Eventually

Oh look, another minimalist portfolio that's so minimal it forgot to include actual content! 

**What we know:**
- Someone named Sneha Rao exists (allegedly)
- They have a portfolio (in theory)
- That's... pretty much it

**The Vibe:** Clean slate energy. So clean there's nothing on it. It's like showing up to an art gallery and finding a single business card taped to the wall. Very "under construction circa 1999" but make it 2024.

**News & Announcements:** The big announcement is that there isn't one. Schrödinger's portfolio—it both exists and doesn't exist until you actually put something in it.

*Rating: 🤷/10 - Would visit again when there's something to actually visit*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">本练习是你第一次体验调用 Frontier Model（前沿大模型）的云端 API。除了后面自己训练/微调模型，课程很多阶段都会用到这类 API（OpenAI、Anthropic 等）。<br/><br/>
更具体地说，这里练的是<strong>摘要（summarization）</strong>——经典 GenAI 用例：新闻摘要、财报摘要、简历/求职信摘要……场景几乎无限。请想想如何在你的业务里做摘要原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续前 — 现在自己试一下</h2>
            <span style="color:#900;">用下面单元格做一个简单的商业小例子。继续围绕摘要：例如粘贴一封邮件正文，让模型建议简短主题行——这正是商业邮件工具里常见的能力。</span>
        </td>
    </tr>
</table>


In [40]:
# ========== 练习脚手架：请你自己填完整（逻辑占位保持原样，勿当已完成代码） ==========

# 第 1 步：创建提示（把 "something here" / 占位正文换成你的业务提示）
system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# 第 2 步：创建消息列表（应类似 [{"role":"system",...},{"role":"user",...}]）
messages = [] # fill this in

# 第三步：调用模型（本笔记本前面用的是 litellm.completion；按你的环境填写）
# 响应=

# 第四步：打印结果
# 打印（


## 额外练习：喜欢网页抓取的同学看这里

你可能发现：若尝试 `display_summary("https://openai.com")` ——往往不行！因为该站大量依赖 JavaScript。常见解法包括 **Selenium**、**Playwright** 等：在后台开浏览器、渲染页面再抽取文本。若你有这类经验，可改进 `Website` / scraper 封装。社区贡献文件夹里有同学提交的 Selenium 示例（谢谢！）。


# 分享你的代码

若你愿意分享改动，作者很乐意转给更多同学！社区贡献文件夹里已有学生改动（含 Selenium）。想加入该目录：提交只含该文件夹新版本的 Pull Request，作者会合并。

若还不熟 git，指南文件夹（指南 3）有完整说明；总览也在：  
https://edwarddonner.com/pr  

提交前请自检：  
1. PR 只包含社区贡献相关改动（除非另有约定）  
2. 笔记本输出清晰  
3. 总计少于约 2000 行，文件也不宜过多  
4. 不要塞不必要的测试文件、过长 README、`.env.example`、表情符号堆砌或其他 LLM 生成物！

非常感谢！

更细步骤示例：  
https://chatgpt.com/share/6873c22b-2a1c-8012-bc9a-debdcf7c835b
